# Week 2 — Deep Learning 입문 + mini UNet 학습 (2조 Intro)

## 이번 주 학습 목표
1. W1 baseline의 한계를 다시 한 번 확인
2. **2D UNet** 이 무엇인지, 왜 보간에 잘 맞는지 이해
3. **mini UNet** (~30K params) 을 본인 노트북에서 직접 학습 (10분~)
4. 학습된 모델의 |Δφ|·SSIM을 W1 Linear baseline과 직접 비교
5. **"학습이 잘 됐는지"** 를 학습 곡선으로 판단하는 법

## 노트북 사용 방법

본 노트북의 모델과 학습 루프는 helpers/model_utils.py에 정의되어 있습니다. 본문에서는 preset과 파라미터를 바꾸어가며 학습 곡선과 평가 지표 변화를 분석합니다. UNet 자체를 처음부터 구현하는 대신, 제공된 모델과 학습 루프를 분석하고 파라미터를 조정해 결과를 평가합니다.

본문 **** 블록에서 preset / k / lr / loss 등을 sweep하면서 학습 곡선과 평가 지표 변화를 관찰. 정답 박스는 없습니다. 본인 노트에 가설·관찰·분석을 자유롭게 기록.

## 0. 환경 준비

In [ ]:
import sys

from pathlib import Path

sys.path.insert(0, str(Path('..').resolve() / 'helpers'))



import numpy as np

import matplotlib.pyplot as plt

import torch



from dr_utils import (

    load_volume, porosity, reconstruct_sparse_linear,

    porosity_error, ssim_3d_mean, summarize_metrics,

    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,

)

from model_utils import (

    UNetMini, count_parameters, SliceDataset,

    train_quick, evaluate_model,

    save_ckpt, load_ckpt, TRAINING_PRESETS,

)

setup_plot_style()



DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'PyTorch {torch.__version__}, device = {DEVICE}')

## 1. W1 baseline 다시 측정 — 우리가 "이길" 대상



본 W2 결과는 "W1 Linear baseline 보다 얼마나 좋아졌나" 로 평가합니다.



> **** Baseline = "비교 기준". Deep learning 결과가 좋다고 말하려면, baseline 보다 얼마나 좋아졌는지 명시해야 합니다.

In [ ]:
DATA = Path('..') / 'data'

bb = load_volume(DATA / 'BB_256.bin')



K = 5  # sparse interval (W1과 동일)

rec_l = reconstruct_sparse_linear(bb, k=K)

m_baseline = summarize_metrics(rec_l, bb, label=f'B1 Linear k={K}')

## 2. UNet 이 무엇인가?



**UNet** 은 의료 영상 / 위성 영상 분할에서 표준으로 쓰이는 신경망 구조.

이름의 "U" 는 모양 (`encoder → bottleneck → decoder`) 이 U자 같아서.



**핵심 3 구성 요소:**

1. **Encoder** — Conv + MaxPool 반복. 영상이 작아지면서 추상 특징 추출.

2. **Decoder** — Conv + ConvTranspose 반복. 크기를 원래로 복원하며 출력 생성.

3. **Skip-connection** — Encoder의 detail(엣지, 텍스처) 을 Decoder에 직접 전달 → 잃어버린 detail 복원.



> **** "앞 슬라이스 + 뒤 슬라이스 → 가운데 슬라이스" 는 "두 이미지 → 한 이미지" 변환. UNet은 입력의 detail 을 잘 보존하면서 출력을 만들 수 있어 적합.

In [ ]:
# 세 preset 의 모델 크기 비교

print(f"{'preset':<10}{'base':>5}{'params':>10}")

for name, p in TRAINING_PRESETS.items():

    m = UNetMini(in_ch=2, base=p['base'])

    print(f'{name:<10}{p["base"]:>5}{count_parameters(m):>10,}')

> **** 위에서 `'fast'` 와 `'standard'` 의 base 채널은 8 vs 16 — 2배. 그런데 파라미터 수는 ~4배 차이.

> 왜 그럴까요? (힌트: Conv layer 파라미터 수 공식 = in × out × kernel²)

## 3. 학습 데이터 만들기 — SliceDataset



**"sparse triplet":** sparse 시나리오에서 모델에게 줄 학습 sample.



한 sample:

- **입력 (input)**: `[slice_before, slice_after]` — 2-channel 이미지

- **출력 (target)**: `slice_middle` — 가운데 슬라이스 (정답, GT)



모델 학습 = "input 으로부터 target 을 예측하는 함수 학습".



> **** 256×256 슬라이스 전체로 학습하면 메모리/시간 부담. 그래서 64×64 작은 patch를 무작위로 잘라 학습. 결과는 동일하게 좋고, 학습은 16배 빠름.

In [ ]:
ds = SliceDataset(bb, k=K, patch_size=64, n_patches_per_triplet=2, augment=True)

print(f'Dataset size: {len(ds)} samples')



x, y = ds[0]

print(f'  하나의 sample: x.shape={x.shape}, y.shape={y.shape}')



# 한 sample 시각화

fig, axes = plt.subplots(1, 3, figsize=(10, 4))

axes[0].imshow(x[0]); axes[0].set_title('입력 ch0: 앞 슬라이스 (before)'); axes[0].axis('off')

axes[1].imshow(x[1]); axes[1].set_title('입력 ch1: 뒤 슬라이스 (after)'); axes[1].axis('off')

axes[2].imshow(y[0]); axes[2].set_title('정답: 가운데 슬라이스 (target)'); axes[2].axis('off')

plt.tight_layout(); plt.show()

## 4. 학습 실행 — `'fast'` preset 으로 ~10분



본 2조용은 `'fast'` 권장 (10분, ~30K params).

1조 형들/누나들이 `'standard'` 나 `'full'` 도 시도하지만, 우리는 먼저 fast로 결과를 한 번 봅시다.

In [ ]:
PRESET = 'fast'  # 권장: 'fast' (10분), 시간 여유 있으면 'standard' (30분)



model, history = train_quick(bb, k=K, preset=PRESET, device=DEVICE, verbose=True)

In [ ]:
# 학습 곡선 — "loss 가 줄어드는가?" 가 학습 잘 됐는지 판단의 1차 기준

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(history, color=ORANGE, lw=2, marker='o')

ax.set_xlabel('Epoch'); ax.set_ylabel('Train L1 loss')

ax.set_title(f'mini UNet 학습 곡선 (preset={PRESET})')

ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()



# 체크포인트 저장

save_ckpt(model, f'unet_mini_{PRESET}.pth',

          meta={'base': TRAINING_PRESETS[PRESET]['base'], 'preset': PRESET, 'k': K})

print(f'\n✓ 모델 저장: unet_mini_{PRESET}.pth')

> **** loss 곡선이 epoch이 지나면서 줄어들었나요? 만약 plateau (평탄) 했다면 무엇이 문제일까요?

> (힌트: 학습률, batch size, 데이터 양 — 세 가지가 가장 흔함)

## 5. 평가 — UNet vs W1 Linear baseline



이제 학습된 모델이 **W1 Linear baseline 보다 얼마나 좋은지** 정량 비교.

In [ ]:
res_unet = evaluate_model(model, bb, k=K, device=DEVICE)



print(f'B1 Linear      |Δφ| = {m_baseline["dphi"]:.2f} %p   SSIM = {m_baseline["ssim"]:.4f}')

print(f'UNet (fast)    |Δφ| = {res_unet["dphi_pp"]:.2f} %p   SSIM = {res_unet["ssim"]:.4f}')



improv = (m_baseline['dphi'] - res_unet['dphi_pp']) / m_baseline['dphi'] * 100

print(f'\n|Δφ| 개선률: {improv:+.1f}%  (양수면 UNet이 더 좋음)')

In [ ]:
# 시각: z=62 (누락) 에서 원본 vs Linear vs UNet

z = 62

recon_unet = res_unet['recon']

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(bb[z]); axes[0].set_title(f'원본 z={z}')

axes[1].imshow(rec_l[z]); axes[1].set_title('B1 Linear')

axes[2].imshow(recon_unet[z]); axes[2].set_title('UNet (fast)')

diff = np.abs(bb[z].astype(float) - recon_unet[z])

axes[3].imshow(diff, cmap='hot'); axes[3].set_title('|원본 − UNet|')

for ax in axes: ax.axis('off')

plt.tight_layout(); plt.show()

> **** UNet 결과가 baseline 보다 명확히 좋아졌나요?

> 만약 미세하게만 좋다면, 어떤 변화를 주면 더 좋아질 것 같나요? (힌트: 학습 더 길게? 모델 더 크게? 데이터 더 많이?)

## 6. 박스 — 파라미터 변경 실험



### PRESET을 'standard' (30분) 로 바꿔 재학습

위 4번 셀의 `PRESET = 'fast'` 를 `'standard'` 로 바꿔 재실행.

**관찰**: 시간이 3배 늘어났을 때 |Δφ| 와 SSIM 이 얼마나 좋아지나요?



### 다른 도메인 평가

BB로 학습한 모델을 CastleGate / Parker 에 평가.

In [ ]:
# 다른 도메인 평가

for name in ['CastleGate', 'Parker']:

    vol = load_volume(DATA / f'{name}_256.bin')

    res = evaluate_model(model, vol, k=K, device=DEVICE)

    print(f'  {name:12s}  UNet |Δφ| = {res["dphi_pp"]:.2f} %p   SSIM = {res["ssim"]:.4f}')

> **** BB 도메인에서 학습한 모델을 다른 도메인 (CG, Parker) 에 적용한 결과의 |Δφ| 가 BB 에서보다 얼마나 떨어졌나요?

> 이 차이를 어떻게 해석할 수 있을까요? (힌트: "학습 데이터 분포" 와 "평가 데이터 분포" 의 차이)

## 7. 다음 주 (W3) — 손실 함수 + HPO 입문

- 본 W2는 L1 loss만 사용 → SSIM, porosity loss 추가하면?
- 학습률·batch size 같은 hyperparameter를 어떻게 고르나?
- Optuna 자동 hyperparameter optimization
- `pip install pytorch-msssim`

---

## 🎯 W2 탐구 과제 (2조 Intro)

다음 과제는 본 노트북의 코드를 수정·확장하며 결과 분석과 함께 정리합니다.

### 과제 1 — preset 비교 (필수)

`'fast'` 와 `'standard'` 두 preset으로 학습 → 시간 / 파라미터 / |Δφ| · SSIM 표 작성. "가성비" 가 좋은 preset이 어느 것인지 본인 기준으로 판단 + 그 근거를 본인 해석으로.

### 과제 2 — Cross-domain 평가 (필수)

학습된 모델을 BB·CastleGate·Parker 세 도메인에 평가. 어느 도메인에서 일반화가 가장 잘 됐나? 왜 그렇게 됐는지 본인 가설 + 시각화로 뒷받침.

### 과제 3 — sparse k 변경 (선택)

K=5 학습 모델을 K=3, K=7 으로도 평가해보고 결과 변화 관찰. 본인의 학습 모델이 어떤 K 범위에서 안정적인지 본인 분석.

### 과제 4 — Learning rate 실험 (선택, 도전)

`train_quick` 내부 `lr=1e-3` 을 1e-4, 5e-3 으로 바꿔보고 학습 곡선 비교. 학습률이 너무 작거나 너무 크면 어떻게 되는지 본인 관찰 + 해석.